# Hurricane Melissa NbS Damage And Service Figures

This notebook creates separate figure exports for the Environmental risks / Hurricane Melissa results section:

- a spatial map of wind thresholds, Melissa track, and benefit-providing NbS areas;
- a damage-by-wind-zone chart;
- an avoided-EAD share-by-wind-zone chart.

The map follows `dphil_papers/agents.md`: imports are kept in the first executable cell, names are explicit, and the Jamaica map includes a north arrow and scale bar. The map uses local annotation helpers so the arrow and scale bar remain stable in the single-panel export.


In [ ]:
from pathlib import Path
import sys

BASE = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
ROBYN_LIBRARY_PATH = BASE / "robyns_libraries"
if str(ROBYN_LIBRARY_PATH) not in sys.path:
    sys.path.append(str(ROBYN_LIBRARY_PATH))

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import Robyn_paper_2_defs
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from IPython.display import display
from rasterio.warp import Resampling, reproject

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [ ]:
PAPER2 = BASE / "dphil_paper_2"
PAPER3 = BASE / "dphil_paper_3"
COMMON = BASE / "dphil_common_cross_cutting"

OUT_DIR = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "nbs_damage_service_figure"
OUT_DIR.mkdir(parents=True, exist_ok=True)

JAMAICA_BOUNDARY_PATH = COMMON / "common_incoming_data" / "boundaries" / "jamaica.gpkg"
MANGROVE_PATCHES_PATH = PAPER3 / "inputs" / "forces_of_nature_mangroves" / "mangroves.shp"
MANGROVE_PATCH_TABLE_PATH = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "mangrove_eads_hurricane_damage" / "mangrove_ead_hurricane_damage_patch_table.csv"
MANGROVE_WIND_SUMMARY_PATH = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "mangrove_eads_hurricane_damage" / "mangrove_ead_hurricane_damage_by_wind_zone_exclusive.csv"
MANGROVE_CHANGE_COUNTS_PATH = PAPER3 / "results" / "threats" / "ndvi" / "draft_processed_images" / "ndvi_mangroves_relative_substantial_change_gt10_map_counts.csv"
FOREST_CHANGE_GT10_PATH = PAPER3 / "results" / "threats" / "ndvi" / "draft_processed_images" / "ndvi_forests_relative_substantial_change_gt10_classes_epsg3448.tif"
FOREST_INCREASE_BY_CLASS_WEIGHTED_PATH = PAPER3 / "results" / "threats" / "ndvi" / "draft_processed_images" / "ndvi_forests_increase_by_landuse_class_weighted.csv"
RIVER_WIND_SUMMARY_PATH = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "river_flood_restoration_eads_hurricane_damage" / "river_flood_restoration_ead_hurricane_damage_by_wind_zone_exclusive.csv"
RIVER_EAD_MIN_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_min.tif"
RIVER_EAD_MAX_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif"
NDVI_BEFORE_PATH = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif"
NDVI_AFTER_PATH = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif"
STORM_TRACK_PATH = PAPER3 / "inputs" / "hurricane_melissa_track_noaa" / "al132025_best_track" / "AL132025_lin.shp"
WIND_SWATH_PATH = PAPER3 / "inputs" / "hurricane_melissa_track_noaa" / "al132025_best_track" / "AL132025_windswath.shp"

J2USD = 1.0 / 150.0
MAP_CRS = "EPSG:3448"
REL_BASELINE_MIN = 0.20
REL_DAMAGE_THRESHOLD = -0.10
REL_GREENING_THRESHOLD = 0.10
FIGURE_DPI = 300
WIND_ZONE_ORDER = [">=64 kt", ">=50 to <64 kt", ">=34 to <50 kt"]
WIND_ZONE_LABELS = [">=64 kt", ">=50 kt", ">=34 kt"]
WIND_THRESHOLD_ORDER = [34.0, 50.0, 64.0]

for input_path in [
    JAMAICA_BOUNDARY_PATH,
    MANGROVE_PATCHES_PATH,
    MANGROVE_PATCH_TABLE_PATH,
    MANGROVE_WIND_SUMMARY_PATH,
    MANGROVE_CHANGE_COUNTS_PATH,
    FOREST_CHANGE_GT10_PATH,
    FOREST_INCREASE_BY_CLASS_WEIGHTED_PATH,
    RIVER_WIND_SUMMARY_PATH,
    RIVER_EAD_MIN_PATH,
    RIVER_EAD_MAX_PATH,
    NDVI_BEFORE_PATH,
    NDVI_AFTER_PATH,
    STORM_TRACK_PATH,
    WIND_SWATH_PATH,
]:
    if not input_path.exists():
        raise FileNotFoundError(input_path)

OUT_DIR


## Helper Functions

In [ ]:
def set_figure_style() -> None:
    """Apply compact plotting defaults for paper figure drafts."""
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 7,
            "axes.titlesize": 8,
            "axes.labelsize": 7,
            "xtick.labelsize": 6.5,
            "ytick.labelsize": 6.5,
            "legend.fontsize": 6,
            "axes.linewidth": 0.6,
            "xtick.major.width": 0.5,
            "ytick.major.width": 0.5,
            "savefig.dpi": FIGURE_DPI,
        }
    )


def clean_geometries(gdf: gpd.GeoDataFrame, target_crs: str) -> gpd.GeoDataFrame:
    """Reproject, repair, and remove empty geometries."""
    clean_gdf = gdf.to_crs(target_crs)
    clean_gdf = clean_gdf[clean_gdf.geometry.notna()].copy()
    clean_gdf = clean_gdf[~clean_gdf.geometry.is_empty].copy()
    clean_gdf["geometry"] = clean_gdf.geometry.make_valid()
    clean_gdf = clean_gdf[~clean_gdf.geometry.is_empty].copy()
    return clean_gdf


def read_noaa_lonlat_layer(path: Path, target_crs: str) -> gpd.GeoDataFrame:
    """Read NOAA best-track layers as lon/lat and reproject to the map CRS."""
    noaa_gdf = gpd.read_file(path)
    noaa_gdf = noaa_gdf.set_crs("EPSG:4326", allow_override=True)
    return clean_geometries(noaa_gdf, target_crs)


def read_positive_ead_usd(path: Path, reference_profile: dict | None = None) -> tuple[np.ndarray, dict]:
    """Read avoided EAD raster, convert JMD to USD, and retain only positive pixels."""
    with rasterio.open(path) as source_raster:
        profile = source_raster.profile.copy()
        ead_array = source_raster.read(1).astype("float64") * J2USD

    ead_array[~np.isfinite(ead_array) | (ead_array <= 0)] = np.nan

    if reference_profile is not None:
        alignment_checks = {
            "crs": profile["crs"] == reference_profile["crs"],
            "transform": profile["transform"] == reference_profile["transform"],
            "height": profile["height"] == reference_profile["height"],
            "width": profile["width"] == reference_profile["width"],
        }
        if not all(alignment_checks.values()):
            raise ValueError(f"Raster alignment mismatch for {path}: {alignment_checks}")

    return ead_array, profile


def reproject_continuous_to_reference(path: Path, reference_profile: dict) -> np.ndarray:
    """Reproject a continuous raster to the river restoration-benefit grid."""
    destination = np.full(
        (reference_profile["height"], reference_profile["width"]),
        np.nan,
        dtype="float32",
    )
    with rasterio.open(path) as source_raster:
        reproject(
            source=rasterio.band(source_raster, 1),
            destination=destination,
            src_transform=source_raster.transform,
            src_crs=source_raster.crs,
            src_nodata=source_raster.nodata,
            dst_transform=reference_profile["transform"],
            dst_crs=reference_profile["crs"],
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return destination


def reproject_categorical_to_reference(path: Path, reference_profile: dict) -> np.ndarray:
    """Reproject a categorical raster to the river restoration-benefit grid."""
    destination = np.full(
        (reference_profile["height"], reference_profile["width"]),
        -9999,
        dtype="int16",
    )
    with rasterio.open(path) as source_raster:
        source_nodata = source_raster.nodata if source_raster.nodata is not None else -9999
        reproject(
            source=rasterio.band(source_raster, 1),
            destination=destination,
            src_transform=source_raster.transform,
            src_crs=source_raster.crs,
            src_nodata=source_nodata,
            dst_transform=reference_profile["transform"],
            dst_crs=reference_profile["crs"],
            dst_nodata=-9999,
            resampling=Resampling.nearest,
        )
    return destination


def pct(numerator: float, denominator: float) -> float:
    """Return percentage, preserving NaN when the denominator is zero."""
    return float(numerator / denominator * 100) if denominator else np.nan


def ead_sum_usd(ead_array: np.ndarray, mask: np.ndarray) -> float:
    """Sum positive avoided EAD values inside a mask."""
    return float(np.nansum(np.where(mask & np.isfinite(ead_array), ead_array, 0.0)))



def add_standard_jamaica_map_furniture(axis: plt.Axes, boundary_gdf: gpd.GeoDataFrame) -> None:
    """Add standard upper-right Jamaica map furniture using shared helpers."""
    scale_bar_point = Robyn_paper_2_defs.add_scale_bar(
        axis,
        boundary_gdf,
        where="right-top",
        pad=0.07,
        length_km=20,
        max_frac=0.22,
        lw=0.5,
        tick_h_frac=0.010,
        fs_lab=5.5,
        fs_unit=5.5,
        unit_text="km",
    )
    if scale_bar_point is None:
        return

    center_data_x, center_data_y = scale_bar_point
    center_axes_x, center_axes_y = axis.transAxes.inverted().transform(
        axis.transData.transform((center_data_x, center_data_y))
    )
    Robyn_paper_2_defs.add_north_arrow_axes(
        axis,
        center_axes_x,
        center_axes_y,
        size_frac=0.060,
        gap_frac=0.030,
        shaft_w_frac=0.10,
        head_w_frac=0.32,
        head_h_frac=0.55,
        fs=5.8,
        lw=0.5,
    )


def save_figure(fig: plt.Figure, stem: str) -> list[Path]:
    """Save one figure to PNG, PDF, and SVG."""
    output_paths = []
    for suffix in ["png", "pdf", "svg"]:
        output_path = OUT_DIR / f"{stem}.{suffix}"
        fig.savefig(output_path, bbox_inches="tight", facecolor="white")
        output_paths.append(output_path)
    return output_paths


set_figure_style()


## Load Spatial Layers And Existing Summary Tables

In [ ]:
jamaica_boundary = clean_geometries(gpd.read_file(JAMAICA_BOUNDARY_PATH), MAP_CRS)
mangrove_patches = clean_geometries(gpd.read_file(MANGROVE_PATCHES_PATH), MAP_CRS)
storm_track = read_noaa_lonlat_layer(STORM_TRACK_PATH, MAP_CRS)
wind_swath = read_noaa_lonlat_layer(WIND_SWATH_PATH, MAP_CRS)
wind_swath_by_threshold = wind_swath.dissolve(by="RADII", as_index=False)

mangrove_patch_table = pd.read_csv(MANGROVE_PATCH_TABLE_PATH)
mangrove_wind_summary = pd.read_csv(MANGROVE_WIND_SUMMARY_PATH)
mangrove_change_counts = pd.read_csv(MANGROVE_CHANGE_COUNTS_PATH)
river_wind_summary = pd.read_csv(RIVER_WIND_SUMMARY_PATH)

mangrove_patches["Mangrove_ID"] = mangrove_patches["ID"].astype(int)
mangrove_benefit_patches = mangrove_patches.merge(
    mangrove_patch_table[
        [
            "Mangrove_ID",
            "positive_avoided_ead_either",
            "damaged_area_ha",
            "mean_damage_after2_minus_before",
        ]
    ],
    on="Mangrove_ID",
    how="left",
)
mangrove_benefit_patches = mangrove_benefit_patches[
    mangrove_benefit_patches["positive_avoided_ead_either"].fillna(False).astype(bool)
].copy()
mangrove_benefit_patches["has_gt10pct_damage"] = mangrove_benefit_patches["damaged_area_ha"].fillna(0) > 0

print(f"Benefit-providing mangrove patches: {len(mangrove_benefit_patches):,}")
print(f"Wind threshold geometries: {len(wind_swath_by_threshold):,}")


## Classify River-Flood Restoration Benefit Pixels By NDVI Response

In [ ]:
river_ead_min_usd, river_profile = read_positive_ead_usd(RIVER_EAD_MIN_PATH)
river_ead_max_usd, _ = read_positive_ead_usd(RIVER_EAD_MAX_PATH, river_profile)
river_transform = river_profile["transform"]
river_shape = (river_profile["height"], river_profile["width"])
river_pixel_area_ha = abs(river_transform.a * river_transform.e) / 10_000
river_benefit_mask = np.isfinite(river_ead_min_usd) | np.isfinite(river_ead_max_usd)

ndvi_before = reproject_continuous_to_reference(NDVI_BEFORE_PATH, river_profile)
ndvi_after = reproject_continuous_to_reference(NDVI_AFTER_PATH, river_profile)
ndvi_eligible_mask = (
    river_benefit_mask
    & np.isfinite(ndvi_before)
    & np.isfinite(ndvi_after)
    & (ndvi_before >= REL_BASELINE_MIN)
)

relative_ndvi_change = np.full(river_shape, np.nan, dtype="float32")
np.divide(
    ndvi_after - ndvi_before,
    ndvi_before,
    out=relative_ndvi_change,
    where=ndvi_eligible_mask,
)

river_damage_mask = ndvi_eligible_mask & (relative_ndvi_change <= REL_DAMAGE_THRESHOLD)
river_greening_mask = ndvi_eligible_mask & (relative_ndvi_change >= REL_GREENING_THRESHOLD)
river_other_mask = river_benefit_mask & ~(river_damage_mask | river_greening_mask)

river_restoration_change_class = np.zeros(river_shape, dtype="uint8")
river_restoration_change_class[river_other_mask] = 1
river_restoration_change_class[river_damage_mask] = 2
river_restoration_change_class[river_greening_mask] = 3
river_restoration_change_class = np.ma.masked_where(river_restoration_change_class == 0, river_restoration_change_class)

river_left, river_bottom, river_right, river_top = rasterio.transform.array_bounds(
    river_profile["height"],
    river_profile["width"],
    river_transform,
)

river_total_benefit_area_ha = river_benefit_mask.sum() * river_pixel_area_ha
river_ndvi_eligible_area_ha = ndvi_eligible_mask.sum() * river_pixel_area_ha
river_damage_area_ha = river_damage_mask.sum() * river_pixel_area_ha
river_greening_area_ha = river_greening_mask.sum() * river_pixel_area_ha
river_other_area_ha = river_other_mask.sum() * river_pixel_area_ha

print(f"River-flood restoration benefit area: {river_total_benefit_area_ha:,.1f} ha")
print(f"NDVI-eligible benefit area: {river_ndvi_eligible_area_ha:,.1f} ha")
print(f"NDVI decrease >10%: {river_damage_area_ha:,.1f} ha")
print(f"NDVI increase >10%: {river_greening_area_ha:,.1f} ha")


## Standardise Wind-Zone And Greening Summaries

In [ ]:
mangrove_wind_plot = mangrove_wind_summary.copy()
mangrove_wind_plot["wind_zone"] = mangrove_wind_plot["wind_zone_exclusive"].replace(
    {
        ">=64 kt": ">=64 kt",
        ">=50 kt": ">=50 to <64 kt",
        ">=34 kt": ">=34 to <50 kt",
    }
)
mangrove_wind_total_ead_min = mangrove_wind_plot["positive_avoided_ead_usd_min"].sum()
mangrove_wind_total_ead_max = mangrove_wind_plot["positive_avoided_ead_usd_max"].sum()
mangrove_wind_plot["nbs_type"] = "Mangroves"
mangrove_wind_plot["benefit_area_ha"] = mangrove_wind_plot["mangrove_area_ha"]
mangrove_wind_plot["damaged_area_ha"] = mangrove_wind_plot["damaged_gt10pct_area_ha"]
mangrove_wind_plot["pct_benefit_area_damaged"] = mangrove_wind_plot["damaged_area_ha"] / mangrove_wind_plot["benefit_area_ha"] * 100
mangrove_wind_plot["positive_avoided_ead_usd_minimum"] = mangrove_wind_plot["positive_avoided_ead_usd_min"]
mangrove_wind_plot["positive_avoided_ead_usd_maximum"] = mangrove_wind_plot["positive_avoided_ead_usd_max"]
mangrove_wind_plot["pct_total_positive_avoided_ead_minimum"] = mangrove_wind_plot["positive_avoided_ead_usd_minimum"] / mangrove_wind_total_ead_min * 100
mangrove_wind_plot["pct_total_positive_avoided_ead_maximum"] = mangrove_wind_plot["positive_avoided_ead_usd_maximum"] / mangrove_wind_total_ead_max * 100

river_wind_plot = river_wind_summary[river_wind_summary["group"].isin(WIND_ZONE_ORDER)].copy()
river_wind_plot = river_wind_plot.rename(columns={"group": "wind_zone"})
river_wind_plot["nbs_type"] = "Forest restoration"

wind_comparison_columns = [
    "nbs_type",
    "wind_zone",
    "benefit_area_ha",
    "damaged_area_ha",
    "pct_benefit_area_damaged",
    "positive_avoided_ead_usd_minimum",
    "positive_avoided_ead_usd_maximum",
    "pct_total_positive_avoided_ead_minimum",
    "pct_total_positive_avoided_ead_maximum",
]
wind_comparison = pd.concat(
    [
        mangrove_wind_plot[wind_comparison_columns],
        river_wind_plot[wind_comparison_columns],
    ],
    ignore_index=True,
)
wind_comparison["wind_zone"] = pd.Categorical(wind_comparison["wind_zone"], categories=WIND_ZONE_ORDER, ordered=True)
wind_comparison = wind_comparison.sort_values(["wind_zone", "nbs_type"]).reset_index(drop=True)
wind_comparison["pct_total_positive_avoided_ead_midpoint"] = (
    wind_comparison["pct_total_positive_avoided_ead_minimum"]
    + wind_comparison["pct_total_positive_avoided_ead_maximum"]
) / 2

mangrove_positive_patch_table = mangrove_patch_table[
    mangrove_patch_table["positive_avoided_ead_either"].astype(bool)
].copy()
mangrove_positive_mean_increase = mangrove_positive_patch_table["mean_damage_after2_minus_before"] > 0
mangrove_positive_mean_decrease = mangrove_positive_patch_table["mean_damage_after2_minus_before"] < 0
mangrove_total_benefit_patch_area_ha = mangrove_positive_patch_table["area_ha"].sum()
mangrove_change_row = mangrove_change_counts.iloc[0]

river_total_ead_min = ead_sum_usd(river_ead_min_usd, np.isfinite(river_ead_min_usd))
river_total_ead_max = ead_sum_usd(river_ead_max_usd, np.isfinite(river_ead_max_usd))
river_greening_ead_min = ead_sum_usd(river_ead_min_usd, river_greening_mask)
river_greening_ead_max = ead_sum_usd(river_ead_max_usd, river_greening_mask)
river_damage_ead_min = ead_sum_usd(river_ead_min_usd, river_damage_mask)
river_damage_ead_max = ead_sum_usd(river_ead_max_usd, river_damage_mask)

greening_summary = pd.DataFrame(
    [
        {
            "nbs_type": "Forest restoration",
            "scope": "river-flood restoration benefit pixels",
            "metric": "relative NDVI change after-before; baseline NDVI >= 0.20; denominator is all benefit pixels",
            "total_area_ha": river_total_benefit_area_ha,
            "ndvi_eligible_area_ha": river_ndvi_eligible_area_ha,
            "decreased_area_ha": river_damage_area_ha,
            "increased_area_ha": river_greening_area_ha,
            "other_area_ha": river_other_area_ha,
            "pct_total_decreased": pct(river_damage_area_ha, river_total_benefit_area_ha),
            "pct_total_increased": pct(river_greening_area_ha, river_total_benefit_area_ha),
            "pct_total_other": pct(river_other_area_ha, river_total_benefit_area_ha),
            "increased_avoided_ead_usd_minimum": river_greening_ead_min,
            "increased_avoided_ead_usd_maximum": river_greening_ead_max,
            "pct_total_avoided_ead_increased_minimum": pct(river_greening_ead_min, river_total_ead_min),
            "pct_total_avoided_ead_increased_maximum": pct(river_greening_ead_max, river_total_ead_max),
            "decreased_avoided_ead_usd_minimum": river_damage_ead_min,
            "decreased_avoided_ead_usd_maximum": river_damage_ead_max,
            "patch_count": np.nan,
            "patch_count_increased": np.nan,
            "total_valid_pixels": np.nan,
            "note": "Exact for river-flood restoration benefit pixels.",
        },
        {
            "nbs_type": "Mangroves",
            "scope": "coastal-flood benefit patches",
            "metric": "patch mean NDVI after-before > 0; not a pixel-level >10% substantial-increase metric",
            "total_area_ha": mangrove_total_benefit_patch_area_ha,
            "ndvi_eligible_area_ha": np.nan,
            "decreased_area_ha": mangrove_positive_patch_table.loc[mangrove_positive_mean_decrease, "area_ha"].sum(),
            "increased_area_ha": mangrove_positive_patch_table.loc[mangrove_positive_mean_increase, "area_ha"].sum(),
            "other_area_ha": np.nan,
            "pct_total_decreased": pct(
                mangrove_positive_patch_table.loc[mangrove_positive_mean_decrease, "area_ha"].sum(),
                mangrove_total_benefit_patch_area_ha,
            ),
            "pct_total_increased": pct(
                mangrove_positive_patch_table.loc[mangrove_positive_mean_increase, "area_ha"].sum(),
                mangrove_total_benefit_patch_area_ha,
            ),
            "pct_total_other": np.nan,
            "increased_avoided_ead_usd_minimum": np.nan,
            "increased_avoided_ead_usd_maximum": np.nan,
            "pct_total_avoided_ead_increased_minimum": np.nan,
            "pct_total_avoided_ead_increased_maximum": np.nan,
            "decreased_avoided_ead_usd_minimum": np.nan,
            "decreased_avoided_ead_usd_maximum": np.nan,
            "patch_count": len(mangrove_positive_patch_table),
            "patch_count_increased": int(mangrove_positive_mean_increase.sum()),
            "total_valid_pixels": np.nan,
            "note": "Existing mangrove avoided-EAD table supports mean patch change, not benefit-patch-specific >10% greening area.",
        },
        {
            "nbs_type": "Mangroves",
            "scope": "all mapped mangrove paired pixels",
            "metric": "relative NDVI increase/decrease >10%; not restricted to benefit patches",
            "total_area_ha": np.nan,
            "ndvi_eligible_area_ha": np.nan,
            "decreased_area_ha": np.nan,
            "increased_area_ha": np.nan,
            "other_area_ha": np.nan,
            "pct_total_decreased": mangrove_change_row["pct_decrease_gt_10pct"],
            "pct_total_increased": mangrove_change_row["pct_increase_gt_10pct"],
            "pct_total_other": mangrove_change_row["pct_other"],
            "increased_avoided_ead_usd_minimum": np.nan,
            "increased_avoided_ead_usd_maximum": np.nan,
            "pct_total_avoided_ead_increased_minimum": np.nan,
            "pct_total_avoided_ead_increased_maximum": np.nan,
            "decreased_avoided_ead_usd_minimum": np.nan,
            "decreased_avoided_ead_usd_maximum": np.nan,
            "patch_count": np.nan,
            "patch_count_increased": np.nan,
            "total_valid_pixels": mangrove_change_row["n_total_valid_paired_mangrove"],
            "note": "Context only: current substantial-change CSV does not retain Mangrove_ID or benefit-patch allocation.",
        },
    ]
)

wind_comparison


## Strict Forest-Equivalent Greening And River Benefit Overlap

In [ ]:
forest_increase_by_class_weighted = pd.read_csv(FOREST_INCREASE_BY_CLASS_WEIGHTED_PATH)
weighted_forest_equivalent_area_ha = forest_increase_by_class_weighted["paired_forest_equiv_area_ha"].sum()
weighted_forest_any_increase_area_ha = forest_increase_by_class_weighted["increased_ndvi_forest_equiv_area_ha"].sum()
weighted_forest_gt10_increase_area_ha = forest_increase_by_class_weighted["substantial_increase_gt10pct_area_ha"].sum()

forest_change_gt10_class = reproject_categorical_to_reference(FOREST_CHANGE_GT10_PATH, river_profile)
forest_gt10_increase_mask = forest_change_gt10_class == 1
forest_gt10_decrease_mask = forest_change_gt10_class == -1
forest_gt10_increase_river_grid_area_ha = forest_gt10_increase_mask.sum() * river_pixel_area_ha

forest_gt10_increase_river_benefit_mask = river_benefit_mask & forest_gt10_increase_mask
forest_gt10_decrease_river_benefit_mask = river_benefit_mask & forest_gt10_decrease_mask

forest_gt10_increase_river_benefit_area_ha = forest_gt10_increase_river_benefit_mask.sum() * river_pixel_area_ha
forest_gt10_decrease_river_benefit_area_ha = forest_gt10_decrease_river_benefit_mask.sum() * river_pixel_area_ha
forest_gt10_increase_river_benefit_ead_min = ead_sum_usd(river_ead_min_usd, forest_gt10_increase_river_benefit_mask)
forest_gt10_increase_river_benefit_ead_max = ead_sum_usd(river_ead_max_usd, forest_gt10_increase_river_benefit_mask)
forest_gt10_decrease_river_benefit_ead_min = ead_sum_usd(river_ead_min_usd, forest_gt10_decrease_river_benefit_mask)
forest_gt10_decrease_river_benefit_ead_max = ead_sum_usd(river_ead_max_usd, forest_gt10_decrease_river_benefit_mask)

forest_greening_overlap_summary = pd.DataFrame(
    [
        {
            "metric": "forest_equivalent_gt10_increase_weighted_all_jamaica",
            "area_ha": weighted_forest_gt10_increase_area_ha,
            "pct_of_weighted_paired_forest_equivalent_area": pct(
                weighted_forest_gt10_increase_area_ha,
                weighted_forest_equivalent_area_ha,
            ),
            "pct_of_weighted_any_ndvi_increase_area": pct(
                weighted_forest_gt10_increase_area_ha,
                weighted_forest_any_increase_area_ha,
            ),
            "positive_avoided_ead_usd_minimum": np.nan,
            "positive_avoided_ead_usd_maximum": np.nan,
            "pct_total_positive_avoided_ead_minimum": np.nan,
            "pct_total_positive_avoided_ead_maximum": np.nan,
            "note": "Weighted forest-equivalent area from 2013 land-cover forest fractions.",
            "pct_of_forest_gt10_increase_on_river_grid_extent": np.nan,
        },
        {
            "metric": "forest_equivalent_gt10_increase_on_river_benefit_grid_extent",
            "area_ha": forest_gt10_increase_river_grid_area_ha,
            "pct_of_weighted_paired_forest_equivalent_area": np.nan,
            "pct_of_weighted_any_ndvi_increase_area": np.nan,
            "positive_avoided_ead_usd_minimum": np.nan,
            "positive_avoided_ead_usd_maximum": np.nan,
            "pct_total_positive_avoided_ead_minimum": np.nan,
            "pct_total_positive_avoided_ead_maximum": np.nan,
            "note": "Unweighted nearest-neighbour forest change raster clipped to river EAD grid extent.",
            "pct_of_forest_gt10_increase_on_river_grid_extent": np.nan,
        },
        {
            "metric": "forest_equivalent_gt10_increase_and_river_benefit_overlap",
            "area_ha": forest_gt10_increase_river_benefit_area_ha,
            "pct_of_weighted_paired_forest_equivalent_area": pct(
                forest_gt10_increase_river_benefit_area_ha,
                weighted_forest_equivalent_area_ha,
            ),
            "pct_of_weighted_any_ndvi_increase_area": pct(
                forest_gt10_increase_river_benefit_area_ha,
                weighted_forest_any_increase_area_ha,
            ),
            "positive_avoided_ead_usd_minimum": forest_gt10_increase_river_benefit_ead_min,
            "positive_avoided_ead_usd_maximum": forest_gt10_increase_river_benefit_ead_max,
            "pct_total_positive_avoided_ead_minimum": pct(forest_gt10_increase_river_benefit_ead_min, river_total_ead_min),
            "pct_total_positive_avoided_ead_maximum": pct(forest_gt10_increase_river_benefit_ead_max, river_total_ead_max),
            "note": "Strict overlap between forest-equivalent >10% NDVI increase raster and positive river-flood restoration avoided-EAD pixels.",
            "pct_of_forest_gt10_increase_on_river_grid_extent": pct(
                forest_gt10_increase_river_benefit_area_ha,
                forest_gt10_increase_river_grid_area_ha,
            ),
        },
        {
            "metric": "forest_equivalent_gt10_decrease_and_river_benefit_overlap_check",
            "area_ha": forest_gt10_decrease_river_benefit_area_ha,
            "pct_of_weighted_paired_forest_equivalent_area": pct(
                forest_gt10_decrease_river_benefit_area_ha,
                weighted_forest_equivalent_area_ha,
            ),
            "pct_of_weighted_any_ndvi_increase_area": np.nan,
            "positive_avoided_ead_usd_minimum": forest_gt10_decrease_river_benefit_ead_min,
            "positive_avoided_ead_usd_maximum": forest_gt10_decrease_river_benefit_ead_max,
            "pct_total_positive_avoided_ead_minimum": pct(forest_gt10_decrease_river_benefit_ead_min, river_total_ead_min),
            "pct_total_positive_avoided_ead_maximum": pct(forest_gt10_decrease_river_benefit_ead_max, river_total_ead_max),
            "note": "Strict overlap check for forest-equivalent >10% NDVI decline on positive river-flood restoration avoided-EAD pixels.",
            "pct_of_forest_gt10_increase_on_river_grid_extent": np.nan,
        },
    ]
)
forest_greening_overlap_summary

## Figure 1: Spatial Exposure And NDVI Response

In [ ]:
nbs_colors = {
    "Mangroves": "#1f78b4",
    "Forest restoration": "#2b8c57",
}
restoration_change_colors = {
    "other": "#d9d9d9",
    "decrease": "#d73027",
    "increase": "#1a9850",
}
restoration_change_cmap = ListedColormap(
    [
        restoration_change_colors["other"],
        restoration_change_colors["decrease"],
        restoration_change_colors["increase"],
    ]
)
restoration_change_norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5], restoration_change_cmap.N)
wind_line_colors = {
    34.0: "#6bb6ff",
    50.0: "#2585d9",
    64.0: "#0057b8",
}

fig, map_axis = plt.subplots(figsize=(180 / 25.4, 118 / 25.4), dpi=FIGURE_DPI)

jamaica_boundary.plot(ax=map_axis, facecolor="#f8f8f8", edgecolor="none", zorder=0)
map_axis.imshow(
    river_restoration_change_class,
    extent=(river_left, river_right, river_bottom, river_top),
    origin="upper",
    cmap=restoration_change_cmap,
    norm=restoration_change_norm,
    interpolation="nearest",
    zorder=2,
)

mangrove_benefit_patches.loc[~mangrove_benefit_patches["has_gt10pct_damage"]].plot(
    ax=map_axis,
    facecolor="#9bd8d2",
    edgecolor="#006d77",
    linewidth=0.18,
    alpha=0.95,
    zorder=4,
)
mangrove_benefit_patches.loc[mangrove_benefit_patches["has_gt10pct_damage"]].plot(
    ax=map_axis,
    facecolor="#08519c",
    edgecolor="#08306b",
    linewidth=0.18,
    alpha=0.95,
    zorder=5,
)

for wind_threshold in WIND_THRESHOLD_ORDER:
    wind_threshold_gdf = wind_swath_by_threshold[wind_swath_by_threshold["RADII"].astype(float).eq(wind_threshold)]
    if not wind_threshold_gdf.empty:
        wind_threshold_gdf.boundary.plot(
            ax=map_axis,
            color=wind_line_colors[wind_threshold],
            linewidth=0.75 if wind_threshold < 64 else 0.95,
            zorder=7,
        )

storm_track.plot(ax=map_axis, color="black", linewidth=0.85, zorder=8)
jamaica_boundary.boundary.plot(ax=map_axis, color="black", linewidth=0.55, zorder=9)

jamaica_min_x, jamaica_min_y, jamaica_max_x, jamaica_max_y = jamaica_boundary.total_bounds
map_axis.set_xlim(jamaica_min_x - (jamaica_max_x - jamaica_min_x) * 0.04, jamaica_max_x + (jamaica_max_x - jamaica_min_x) * 0.04)
map_axis.set_ylim(jamaica_min_y - (jamaica_max_y - jamaica_min_y) * 0.11, jamaica_max_y + (jamaica_max_y - jamaica_min_y) * 0.10)
map_axis.set_axis_off()
map_axis.set_title("Benefit-providing NbS exposure and NDVI response after Hurricane Melissa", pad=4)

add_standard_jamaica_map_furniture(map_axis, jamaica_boundary)

map_handles = [
    mpatches.Patch(facecolor=restoration_change_colors["decrease"], edgecolor="none", label="Forest restoration benefit: NDVI decrease >10%"),
    mpatches.Patch(facecolor=restoration_change_colors["increase"], edgecolor="none", label="Forest restoration benefit: NDVI increase >10%"),
    mpatches.Patch(facecolor=restoration_change_colors["other"], edgecolor="none", label="Forest restoration benefit: <10% NDVI change or not NDVI eligible"),
    mpatches.Patch(facecolor="#9bd8d2", edgecolor="#006d77", label="Mangrove benefit patch"),
    mpatches.Patch(facecolor="#08519c", edgecolor="#08306b", label="Damaged mangrove benefit patch"),
    Line2D([0], [0], color=wind_line_colors[34.0], linewidth=0.8, label="34 kt wind threshold"),
    Line2D([0], [0], color=wind_line_colors[50.0], linewidth=0.8, label="50 kt wind threshold"),
    Line2D([0], [0], color=wind_line_colors[64.0], linewidth=1.0, label="64 kt wind threshold"),
    Line2D([0], [0], color="black", linewidth=0.85, label="Melissa track"),
]
map_axis.legend(
    handles=map_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.035),
    ncol=3,
    frameon=True,
    framealpha=1.0,
    facecolor="white",
    edgecolor="#cfcfcf",
    borderpad=0.45,
    handlelength=1.4,
    columnspacing=1.0,
)
fig.subplots_adjust(bottom=0.22)
map_figure_paths = save_figure(fig, "hurricane_melissa_nbs_damage_service_map")
display(fig)
plt.close(fig)

map_figure_paths


## Figure 2: Damage By Wind Zone

In [ ]:
fig, damage_axis = plt.subplots(figsize=(90 / 25.4, 72 / 25.4), dpi=FIGURE_DPI)
bar_width = 0.36
zone_positions = np.arange(len(WIND_ZONE_ORDER))

for nbs_index, nbs_type in enumerate(["Mangroves", "Forest restoration"]):
    nbs_rows = wind_comparison[wind_comparison["nbs_type"].eq(nbs_type)].set_index("wind_zone").loc[WIND_ZONE_ORDER]
    bar_offset = (nbs_index - 0.5) * bar_width
    damage_axis.bar(
        zone_positions + bar_offset,
        nbs_rows["pct_benefit_area_damaged"],
        width=bar_width,
        color=nbs_colors[nbs_type],
        label=nbs_type,
    )

for axis_position, wind_zone in zip(zone_positions, WIND_ZONE_ORDER, strict=True):
    zone_max = wind_comparison.loc[wind_comparison["wind_zone"].eq(wind_zone), "pct_benefit_area_damaged"].max()
    damage_axis.text(axis_position, zone_max + 2.0, f"{zone_max:.0f}%", ha="center", va="bottom", fontsize=6)

damage_axis.set_xticks(zone_positions)
damage_axis.set_xticklabels(WIND_ZONE_LABELS)
damage_axis.set_ylabel("Benefit area damaged (%)")
damage_axis.set_ylim(0, 80)
damage_axis.grid(axis="y", linewidth=0.35, alpha=0.35)
damage_axis.legend(frameon=False, loc="upper right")
damage_axis.set_title("Damage increased with wind exposure", pad=4)
damage_axis.spines["top"].set_visible(False)
damage_axis.spines["right"].set_visible(False)
fig.tight_layout()
damage_figure_paths = save_figure(fig, "hurricane_melissa_nbs_damage_by_wind_zone")
display(fig)
plt.close(fig)

damage_figure_paths


## Figure 3: Avoided EAD By Wind Zone

In [ ]:
fig, ead_axis = plt.subplots(figsize=(90 / 25.4, 72 / 25.4), dpi=FIGURE_DPI)
point_width = 0.26

for nbs_index, nbs_type in enumerate(["Mangroves", "Forest restoration"]):
    nbs_rows = wind_comparison[wind_comparison["nbs_type"].eq(nbs_type)].set_index("wind_zone").loc[WIND_ZONE_ORDER]
    point_offset = (nbs_index - 0.5) * point_width
    midpoint = nbs_rows["pct_total_positive_avoided_ead_midpoint"].to_numpy()
    lower_error = midpoint - nbs_rows["pct_total_positive_avoided_ead_minimum"].to_numpy()
    upper_error = nbs_rows["pct_total_positive_avoided_ead_maximum"].to_numpy() - midpoint
    ead_axis.errorbar(
        zone_positions + point_offset,
        midpoint,
        yerr=np.vstack([np.abs(lower_error), np.abs(upper_error)]),
        fmt="o",
        color=nbs_colors[nbs_type],
        markerfacecolor="white",
        markeredgewidth=1.0,
        capsize=2.8,
        linewidth=1.0,
        label=nbs_type,
    )

ead_axis.set_xticks(zone_positions)
ead_axis.set_xticklabels(WIND_ZONE_LABELS)
ead_axis.set_ylabel("Total positive avoided EAD in wind class (%)")
ead_axis.set_ylim(0, 95)
ead_axis.grid(axis="y", linewidth=0.35, alpha=0.35)
ead_axis.legend(frameon=False, loc="upper right")
ead_axis.set_title("Service value was concentrated differently", pad=4)
ead_axis.spines["top"].set_visible(False)
ead_axis.spines["right"].set_visible(False)
fig.tight_layout()
ead_figure_paths = save_figure(fig, "hurricane_melissa_nbs_avoided_ead_by_wind_zone")
display(fig)
plt.close(fig)

ead_figure_paths


## Export Data And Metadata

In [ ]:
wind_summary_path = OUT_DIR / "hurricane_melissa_nbs_damage_service_wind_summary.csv"
greening_summary_path = OUT_DIR / "hurricane_melissa_nbs_greening_summary.csv"
forest_greening_overlap_summary_path = OUT_DIR / "forest_greening_river_benefit_overlap_summary.csv"
metadata_path = OUT_DIR / "hurricane_melissa_nbs_damage_service_method_metadata.csv"

figure_metadata = pd.DataFrame(
    [
        {"name": "jamaica_boundary_path", "value": str(JAMAICA_BOUNDARY_PATH)},
        {"name": "mangrove_patches_path", "value": str(MANGROVE_PATCHES_PATH)},
        {"name": "mangrove_patch_table_path", "value": str(MANGROVE_PATCH_TABLE_PATH)},
        {"name": "mangrove_wind_summary_path", "value": str(MANGROVE_WIND_SUMMARY_PATH)},
        {"name": "mangrove_change_counts_path", "value": str(MANGROVE_CHANGE_COUNTS_PATH)},
        {"name": "forest_change_gt10_path", "value": str(FOREST_CHANGE_GT10_PATH)},
        {"name": "forest_increase_by_class_weighted_path", "value": str(FOREST_INCREASE_BY_CLASS_WEIGHTED_PATH)},
        {"name": "river_wind_summary_path", "value": str(RIVER_WIND_SUMMARY_PATH)},
        {"name": "river_ead_min_path", "value": str(RIVER_EAD_MIN_PATH)},
        {"name": "river_ead_max_path", "value": str(RIVER_EAD_MAX_PATH)},
        {"name": "ndvi_before_path", "value": str(NDVI_BEFORE_PATH)},
        {"name": "ndvi_after_path", "value": str(NDVI_AFTER_PATH)},
        {"name": "storm_track_path", "value": str(STORM_TRACK_PATH)},
        {"name": "wind_swath_path", "value": str(WIND_SWATH_PATH)},
        {"name": "map_crs", "value": MAP_CRS},
        {"name": "currency_conversion", "value": "JMD to USD = 1/150 for river-flood avoided EAD rasters"},
        {"name": "damage_threshold", "value": "relative NDVI decline greater than or equal to 10%"},
        {"name": "greening_threshold", "value": "relative NDVI increase greater than or equal to 10%"},
        {"name": "ndvi_baseline_min", "value": str(REL_BASELINE_MIN)},
        {"name": "wind_zone_plot_order", "value": "; ".join(WIND_ZONE_ORDER)},
        {"name": "north_arrow_and_scale_bar", "value": "shared Robyn_paper_2_defs helpers, right-top placement"},
    ]
)

wind_comparison.to_csv(wind_summary_path, index=False)
greening_summary.to_csv(greening_summary_path, index=False)
forest_greening_overlap_summary.to_csv(forest_greening_overlap_summary_path, index=False)
figure_metadata.to_csv(metadata_path, index=False)

all_output_paths = (
    map_figure_paths
    + damage_figure_paths
    + ead_figure_paths
    + [wind_summary_path, greening_summary_path, forest_greening_overlap_summary_path, metadata_path]
)
all_output_paths


## Key Greening Results

In [ ]:
display(
    greening_summary[
        [
            "nbs_type",
            "scope",
            "metric",
            "total_area_ha",
            "increased_area_ha",
            "pct_total_increased",
            "increased_avoided_ead_usd_minimum",
            "increased_avoided_ead_usd_maximum",
            "patch_count",
            "patch_count_increased",
            "total_valid_pixels",
            "note",
        ]
    ]
)

display(forest_greening_overlap_summary)